# LFortran - 1 Hz Square Wave

Created with Grok Build

A one-second square wave at 1 Hz with 50% duty, switching between -1 and +1, sampled 1024 times. A square wave of this shape is a sum of odd sine harmonics, so the lines fall at 1, 3, 5, … Hz and the magnitude at harmonic $k$ is near $4/(k\pi)$. The phase of each of those lines is near -90 degrees.

The FFTW3 page's title and SciLAB listing are 1 Hz. Its introduction says 11 Hz. This page follows the listing. The time plot is the whole second, one period. The spectrum plot runs out to 40 Hz, which includes the harmonics still above magnitude 0.05.


The first code cell is `signals.f90` followed by `spectrum.f90`. The next cell builds this signal and prints the bins at or above magnitude 0.05.


In [1]:
module signals
    use, intrinsic :: iso_fortran_env, only: int32, real64
    implicit none
    private
    public :: sine_tone, square_wave, two_sines, switched_sine

contains

    function sine_tone(n, sample_rate, frequency, amplitude) result(samples)
        integer(int32), intent(in) :: n
        real(real64), intent(in) :: sample_rate
        real(real64), intent(in) :: frequency
        real(real64), intent(in) :: amplitude
        real(real64), allocatable :: samples(:)
        integer(int32) :: i
        real(real64) :: pi

        pi = acos(-1.0_real64)
        allocate(samples(n))
        do i = 1, n
            samples(i) = amplitude * sin(2.0_real64 * pi * frequency * time_at(i, sample_rate))
        end do
    end function sine_tone

    function square_wave(n, sample_rate, frequency, amplitude) result(samples)
        integer(int32), intent(in) :: n
        real(real64), intent(in) :: sample_rate
        real(real64), intent(in) :: frequency
        real(real64), intent(in) :: amplitude
        real(real64), allocatable :: samples(:)
        integer(int32) :: i
        real(real64) :: position

        allocate(samples(n))
        do i = 1, n
            position = modulo(time_at(i, sample_rate) * frequency, 1.0_real64)
            samples(i) = merge(amplitude, -amplitude, position < 0.5_real64)
        end do
    end function square_wave

    function two_sines(n, sample_rate, frequency_a, amplitude_a, &
            frequency_b, amplitude_b) result(samples)
        integer(int32), intent(in) :: n
        real(real64), intent(in) :: sample_rate
        real(real64), intent(in) :: frequency_a
        real(real64), intent(in) :: amplitude_a
        real(real64), intent(in) :: frequency_b
        real(real64), intent(in) :: amplitude_b
        real(real64), allocatable :: samples(:)
        integer(int32) :: i
        real(real64) :: pi, t

        pi = acos(-1.0_real64)
        allocate(samples(n))
        do i = 1, n
            t = time_at(i, sample_rate)
            samples(i) = amplitude_a * sin(2.0_real64 * pi * frequency_a * t) &
                + amplitude_b * sin(2.0_real64 * pi * frequency_b * t)
        end do
    end function two_sines

    function switched_sine(n, sample_rate, carrier, gate, amplitude) result(samples)
        integer(int32), intent(in) :: n
        real(real64), intent(in) :: sample_rate
        real(real64), intent(in) :: carrier
        real(real64), intent(in) :: gate
        real(real64), intent(in) :: amplitude
        real(real64), allocatable :: samples(:)
        integer(int32) :: i
        real(real64) :: pi, t, position

        pi = acos(-1.0_real64)
        allocate(samples(n))
        do i = 1, n
            t = time_at(i, sample_rate)
            position = modulo(t * gate, 1.0_real64)
            samples(i) = merge(amplitude * sin(2.0_real64 * pi * carrier * t), &
                0.0_real64, position < 0.5_real64)
        end do
    end function switched_sine

    pure real(real64) function time_at(index, sample_rate)
        integer(int32), intent(in) :: index
        real(real64), intent(in) :: sample_rate
        time_at = real(index - 1, real64) / sample_rate
    end function time_at

end module signals

module spectrum
    use, intrinsic :: iso_fortran_env, only: int32, real64
    implicit none
    private
    public :: spectrum_of, write_series, print_peaks

contains

    subroutine spectrum_of(samples, sample_rate, magnitude, frequency, phase)
        real(real64), intent(in) :: samples(:)
        real(real64), intent(in) :: sample_rate
        real(real64), allocatable, intent(out) :: magnitude(:)
        real(real64), allocatable, intent(out) :: frequency(:)
        real(real64), allocatable, intent(out) :: phase(:)
        complex(real64), allocatable :: bins(:)
        integer(int32) :: n, k, half
        real(real64) :: pi, scale

        n = size(samples, kind=int32)
        if (.not. is_power_of_two(n)) then
            error stop 'spectrum_of: length must be a power of two'
        end if

        allocate(bins(n))
        do k = 1, n
            bins(k) = cmplx(samples(k), 0.0_real64, real64)
        end do
        call fft_inplace(bins)

        half = n / 2
        allocate(magnitude(half + 1))
        allocate(frequency(half + 1))
        allocate(phase(half + 1))
        pi = acos(-1.0_real64)
        scale = real(n, real64)
        do k = 0, half
            frequency(k + 1) = real(k, real64) * sample_rate / scale
            if (k == 0 .or. k == half) then
                magnitude(k + 1) = abs(bins(k + 1)) / scale
            else
                magnitude(k + 1) = 2.0_real64 * abs(bins(k + 1)) / scale
            end if
            phase(k + 1) = atan2(aimag(bins(k + 1)), real(bins(k + 1))) * (180.0_real64 / pi)
        end do
    end subroutine spectrum_of

    subroutine write_series(time_path, spectrum_path, sample_rate, samples, &
            magnitude, frequency, phase)
        character(len=*), intent(in) :: time_path
        character(len=*), intent(in) :: spectrum_path
        real(real64), intent(in) :: sample_rate
        real(real64), intent(in) :: samples(:)
        real(real64), intent(in) :: magnitude(:)
        real(real64), intent(in) :: frequency(:)
        real(real64), intent(in) :: phase(:)
        integer(int32) :: unit, i

        open(newunit=unit, file=time_path, status='replace', action='write')
        do i = 1, size(samples, kind=int32)
            write(unit, '(es24.16, 1x, es24.16)') &
                real(i - 1, real64) / sample_rate, samples(i)
        end do
        close(unit)

        open(newunit=unit, file=spectrum_path, status='replace', action='write')
        do i = 1, size(magnitude, kind=int32)
            write(unit, '(es24.16, 1x, es24.16, 1x, es24.16)') &
                frequency(i), magnitude(i), phase(i)
        end do
        close(unit)
    end subroutine write_series

    subroutine print_peaks(frequency, magnitude, phase, threshold)
        real(real64), intent(in) :: frequency(:)
        real(real64), intent(in) :: magnitude(:)
        real(real64), intent(in) :: phase(:)
        real(real64), intent(in) :: threshold
        integer(int32) :: i, count

        count = size(magnitude, kind=int32)
        print '(a)', 'frequency_hz magnitude phase_deg'
        do i = 0, count - 1
            call print_one_peak(frequency(i + 1), magnitude(i + 1), phase(i + 1), threshold)
        end do
    end subroutine print_peaks

    ! A block-if on a real comparison is miscompiled by the LFortran kernel.
    ! The test is stored in a logical, and the branch uses that logical.
    subroutine print_one_peak(frequency, magnitude, phase, threshold)
        real(real64), intent(in) :: frequency
        real(real64), intent(in) :: magnitude
        real(real64), intent(in) :: phase
        real(real64), intent(in) :: threshold
        character(len=48) :: line
        logical :: keep

        keep = magnitude >= threshold
        write(line, '(f12.3, 1x, f12.6, 1x, f12.3)') frequency, magnitude, phase
        if (.not. keep) return
        print '(a)', trim(line)
    end subroutine print_one_peak

    subroutine fft_inplace(bins)
        complex(real64), intent(inout) :: bins(:)
        complex(real64) :: twiddle, twiddle_step, even, odd
        integer(int32) :: n, bits, i, reversed, length, half, start, k
        real(real64) :: angle

        n = size(bins, kind=int32)
        bits = 0
        length = 1
        do while (length < n)
            length = length * 2
            bits = bits + 1
        end do

        do i = 0, n - 1
            reversed = reverse_bits(i, bits)
            if (reversed > i) then
                even = bins(i + 1)
                bins(i + 1) = bins(reversed + 1)
                bins(reversed + 1) = even
            end if
        end do

        length = 2
        do while (length <= n)
            half = length / 2
            angle = -2.0_real64 * acos(-1.0_real64) / real(length, real64)
            twiddle_step = cmplx(cos(angle), sin(angle), real64)
            do start = 1, n, length
                twiddle = cmplx(1.0_real64, 0.0_real64, real64)
                do k = 0, half - 1
                    even = bins(start + k)
                    odd = twiddle * bins(start + k + half)
                    bins(start + k) = even + odd
                    bins(start + k + half) = even - odd
                    twiddle = twiddle * twiddle_step
                end do
            end do
            length = length * 2
        end do
    end subroutine fft_inplace

    pure integer(int32) function reverse_bits(value, bits) result(reversed)
        integer(int32), intent(in) :: value
        integer(int32), intent(in) :: bits
        integer(int32) :: bit, remaining

        reversed = 0
        remaining = value
        do bit = 1, bits
            reversed = reversed * 2_int32 + mod(remaining, 2_int32)
            remaining = remaining / 2_int32
        end do
    end function reverse_bits

    pure logical function is_power_of_two(n)
        integer(int32), intent(in) :: n
        is_power_of_two = n >= 2 .and. iand(n, n - 1_int32) == 0
    end function is_power_of_two

end module spectrum


In [2]:
use, intrinsic :: iso_fortran_env, only: real64
use signals
use spectrum

real(real64), allocatable :: samples(:)
real(real64), allocatable :: magnitude(:)
real(real64), allocatable :: frequency(:)
real(real64), allocatable :: phase(:)

samples = square_wave(1024, 1024.0_real64, 1.0_real64, 1.0_real64)
call spectrum_of(samples, 1024.0_real64, magnitude, frequency, phase)
call write_series('square_1_hz_time.dat', 'square_1_hz_spectrum.dat', &
    1024.0_real64, samples, magnitude, frequency, phase)
call print_peaks(frequency, magnitude, phase, 0.05_real64)


frequency_hz magnitude phase_deg
       1.000     1.273242      -89.824
       3.000     0.424419      -89.473
       5.000     0.254658      -89.121
       7.000     0.181905      -88.770
       9.000     0.141489      -88.418
      11.000     0.115771      -88.066
      13.000     0.097967      -87.715
      15.000     0.084913      -87.363
      17.000     0.074930      -87.012
      19.000     0.067051      -86.660
      21.000     0.060672      -86.309
      23.000     0.055404      -85.957
      25.000     0.050980      -85.605


:::{figure} images/square_1_hz_time.png
:class: clickable-figure

One period of a 1 Hz square wave, 50% duty, sampled at 1024 Hz.
:::

:::{figure} images/square_1_hz_spectrum.png
:class: clickable-figure

Magnitude spectrum of the square wave. The lines are the odd harmonics of 1 Hz.
:::
